# Avance Fase 3 — Semana 2

## Núcleo algorítmico y programación orientada a objetos

**Caso de trabajo: predicción de accidente cerebrovascular (*stroke*)**

---

En la Fase 2 ustedes construyeron un pipeline que funciona. Este cuaderno lo toma y le
da forma de **sistema**: las mismas operaciones, ahora organizadas en clases que se
pueden probar, reemplazar y reutilizar.

No hay datos nuevos ni pipeline nuevo. **Lo que cambia es la arquitectura del código.**

### Recorrido

| Parte | Contenido | Criterio de la rúbrica que alimenta |
|---|---|---|
| 1 | De funciones a objetos: por qué | Codificación funcional |
| 2 | Los cinco conceptos de la POO | Programación orientada a objetos |
| 3 | La clase `Preprocesador` | Preprocesamiento y transformación |
| 4 | Encapsulamiento: proteger el estado | Programación orientada a objetos |
| 5 | Herencia y polimorfismo | Programación orientada a objetos |
| 6 | El `Pipeline`: componer los pasos | Diseño estructurado |
| 7 | Cohesión y acoplamiento | Diseño estructurado |
| 8 | Validación: casos normales, límite y excepciones | Validación técnica |
| 9 | Recursividad | Diseño estructurado |
| 10 | Eficiencia: medir tiempo y memoria | Eficiencia y optimización |
| 11 | Patrones de diseño | Documentación de arquitectura |
| 12 | Adaptar el cuaderno a su propio conjunto | Codificación funcional |
| 13 | El resultado: código y datos listos para la Fase 4 | Documentación de arquitectura |
| 14 | Ejercicios para su proyecto | Todos |

**Cómo usarlo.** Los datos se ingresan **desde fuera**: el cuaderno lee el archivo que
ustedes indiquen en la celda de configuración.

---
## Configuración

**Esta es la única celda que hay que editar.** Todo lo que sigue se construye a partir
de ella.

In [1]:
# =====================================================================
# CONFIGURACIÓN DEL PROYECTO
# Editen solo esta celda para trabajar con su propio conjunto de datos
# =====================================================================

RUTA_DATOS = "data/raw/healthcare-dataset-stroke-data.csv"

COLUMNA_OBJETIVO = "stroke"          # la variable que se quiere explicar
COLUMNA_ID = "id"                    # identificador: se elimina del análisis

COLUMNAS_CONTINUAS = ["age", "avg_glucose_level", "bmi"]
COLUMNAS_NOMINALES = ["gender", "work_type", "smoking_status", "Residence_type",
                      "ever_married"]
COLUMNAS_BINARIAS = ["hypertension", "heart_disease"]

SEMILLA = 42
PROPORCION_PRUEBA = 0.2

# Si el archivo no está disponible, permite ejecutar con datos de demostración
# del mismo esquema. Pónganlo en False cuando trabajen con su archivo real.
PERMITIR_DEMOSTRACION = True
# =====================================================================

COLUMNAS_ESPERADAS = ([COLUMNA_ID, COLUMNA_OBJETIVO] + COLUMNAS_CONTINUAS
                      + COLUMNAS_NOMINALES + COLUMNAS_BINARIAS)

print("Archivo configurado :", RUTA_DATOS)
print("Variable objetivo   :", COLUMNA_OBJETIVO)
print("Columnas esperadas  :", len(COLUMNAS_ESPERADAS))

Archivo configurado : data/raw/healthcare-dataset-stroke-data.csv
Variable objetivo   : stroke
Columnas esperadas  : 12


## Preparación del entorno

In [8]:
import os
import time
import timeit
import tracemalloc

import numpy as np
import pandas as pd

np.random.seed(SEMILLA)

print("pandas :", pd.__version__)
print("NumPy  :", np.__version__)
print("Semilla:", SEMILLA)

pandas : 3.0.6
NumPy  : 2.5.3
Semilla: 42


## Carga del conjunto desde el archivo externo

El cuaderno **no genera los datos**: los lee del archivo que ustedes indiquen en la
configuración. La función de carga hace tres cosas antes de devolver nada:

1. Comprueba que el archivo exista, y si no, explica dónde ponerlo.
2. Lo lee según su extensión, sea CSV o Excel.
3. Verifica que estén las columnas declaradas en la configuración.

Ese tercer paso es el que evita el error más molesto: descubrir en la celda veinte que
una columna se llamaba distinto.

In [9]:
def leer_archivo(ruta):
    """Lee el archivo según su extensión y devuelve un DataFrame."""
    extension = os.path.splitext(ruta)[1].lower()

    if extension in (".csv", ".txt"):
        # sep=None con engine="python" infiere el separador: útil con datos
        # públicos chilenos, que suelen venir con punto y coma
        return pd.read_csv(ruta, sep=None, engine="python", encoding="utf-8")
    if extension in (".xlsx", ".xls"):
        return pd.read_excel(ruta)
    if extension == ".parquet":
        return pd.read_parquet(ruta)

    raise ValueError(
        f"Extensión no reconocida: '{extension}'. "
        "Se admiten .csv, .txt, .xlsx, .xls y .parquet."
    )


def verificar_esquema(df, columnas_esperadas):
    """Comprueba que estén todas las columnas declaradas en la configuración."""
    faltantes = [c for c in columnas_esperadas if c not in df.columns]
    if faltantes:
        raise KeyError(
            f"Faltan columnas declaradas en la configuración: {faltantes}\n"
            f"Columnas disponibles en el archivo: {list(df.columns)}"
        )
    sobrantes = [c for c in df.columns if c not in columnas_esperadas]
    return {"declaradas": len(columnas_esperadas),
            "en_archivo": df.shape[1],
            "no_declaradas": sobrantes}

In [10]:
def generar_demostracion(n=5110, semilla=SEMILLA):
    """Datos con el mismo esquema, solo para que el cuaderno corra sin el archivo.

    NO usar en el proyecto: sirve para revisar el cuaderno antes de tener
    los datos a mano.
    """
    rng = np.random.default_rng(semilla)
    df = pd.DataFrame({
        "id": rng.choice(np.arange(1, n * 20), size=n, replace=False),
        "gender": rng.choice(["Male", "Female", "Other"], n, p=[0.41, 0.585, 0.005]),
        "age": np.round(rng.uniform(0.08, 82, n), 1),
        "hypertension": rng.choice([0, 1], n, p=[0.90, 0.10]),
        "heart_disease": rng.choice([0, 1], n, p=[0.95, 0.05]),
        "ever_married": rng.choice(["Yes", "No"], n, p=[0.66, 0.34]),
        "work_type": rng.choice(
            ["Private", "Self-employed", "Govt_job", "children", "Never_worked"],
            n, p=[0.57, 0.16, 0.13, 0.13, 0.01]),
        "Residence_type": rng.choice(["Urban", "Rural"], n, p=[0.51, 0.49]),
        "avg_glucose_level": np.round(rng.uniform(55, 272, n), 2),
        "bmi": np.round(rng.normal(28.9, 7.9, n), 1),
        "smoking_status": rng.choice(
            ["never smoked", "Unknown", "formerly smoked", "smokes"],
            n, p=[0.37, 0.30, 0.17, 0.16]),
        "stroke": rng.choice([0, 1], n, p=[0.951, 0.049]),
    })
    df.loc[rng.choice(df.index, size=201, replace=False), "bmi"] = np.nan
    return df


def cargar(ruta=RUTA_DATOS, columnas_esperadas=None, permitir_demo=PERMITIR_DEMOSTRACION):
    """Carga el conjunto desde el archivo externo y verifica su esquema."""
    columnas_esperadas = columnas_esperadas or COLUMNAS_ESPERADAS

    if os.path.exists(ruta):
        df = leer_archivo(ruta)
        print(f"Archivo leído: {ruta}")
    elif permitir_demo:
        df = generar_demostracion()
        print("AVISO: no se encontró el archivo. Se usan DATOS DE DEMOSTRACIÓN.")
        print(f"       Coloquen su archivo en: {ruta}")
        print("       Y pongan PERMITIR_DEMOSTRACION = False en la configuración.")
    else:
        raise FileNotFoundError(
            f"No se encontró el archivo: {ruta}\n"
            f"Carpeta actual: {os.getcwd()}\n"
            "Revisen la variable RUTA_DATOS en la celda de configuración. "
            "La ruta se escribe desde la carpeta donde está este cuaderno."
        )

    # Las columnas numéricas se fuerzan a número: un texto suelto las
    # convertiría en columna de objetos sin ningún aviso
    for columna in COLUMNAS_CONTINUAS:
        if columna in df.columns:
            df[columna] = pd.to_numeric(df[columna], errors="coerce")

    informe = verificar_esquema(df, columnas_esperadas)
    print(f"Forma: {df.shape[0]} filas x {df.shape[1]} columnas")
    print(f"Esquema verificado: {informe['declaradas']} columnas declaradas, "
          f"{len(informe['no_declaradas'])} no declaradas")
    if informe["no_declaradas"]:
        print("  No declaradas:", informe["no_declaradas"])
    return df


datos = cargar()
datos.head()

AVISO: no se encontró el archivo. Se usan DATOS DE DEMOSTRACIÓN.
       Coloquen su archivo en: data/raw/healthcare-dataset-stroke-data.csv
       Y pongan PERMITIR_DEMOSTRACION = False en la configuración.
Forma: 5110 filas x 12 columnas
Esquema verificado: 12 columnas declaradas, 0 no declaradas


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,27693,Female,37.0,0,0,Yes,Private,Rural,116.71,37.1,formerly smoked,0
1,31631,Male,63.3,0,0,Yes,Private,Urban,181.10,24.7,Unknown,0
2,92238,Male,30.2,0,0,No,Private,Rural,68.50,25.9,never smoked,0
3,73628,Female,51.9,0,0,Yes,Private,Rural,59.33,24.2,smokes,0
4,62504,Male,42.0,0,0,No,Private,Rural,182.57,22.9,never smoked,0


**Si obtienen un `KeyError` al cargar**, la configuración no coincide con el archivo.
El mensaje lista las columnas que faltan y las que sí están: corrijan los nombres en la
celda de configuración y vuelvan a ejecutar.

**Si el cuaderno avisa que usa datos de demostración**, revisen la ruta. Los números que
verán después no serán los de su proyecto.

## El conjunto antes de tocarlo

Un perfil que funciona con cualquier conjunto, no solo con este. Se apoya en la
configuración, así que al cambiar de proyecto sigue sirviendo.

In [11]:
def perfilar(df):
    """Devuelve una fila por columna con su tipo, únicos y porcentaje de nulos."""
    return pd.DataFrame({
        "columna": df.columns,
        "tipo": [str(t) for t in df.dtypes],
        "unicos": [df[c].nunique(dropna=True) for c in df.columns],
        "nulos": df.isna().sum().values,
        "pct_nulos": (df.isna().mean() * 100).round(2).values,
    }).sort_values("pct_nulos", ascending=False).reset_index(drop=True)


perfil = perfilar(datos)
print("Columnas con valores faltantes:")
print(perfil[perfil["nulos"] > 0].to_string(index=False))

print(f"\nDistribución de la variable objetivo '{COLUMNA_OBJETIVO}':")
print(datos[COLUMNA_OBJETIVO].value_counts(normalize=True).round(4))

print("\nCategorías de smoking_status (caso stroke):")
print(datos["smoking_status"].value_counts())

Columnas con valores faltantes:
columna    tipo  unicos  nulos  pct_nulos
    bmi float64     428    201       3.93

Distribución de la variable objetivo 'stroke':
stroke
0    0.9474
1    0.0526
Name: proportion, dtype: float64

Categorías de smoking_status (caso stroke):
smoking_status
never smoked       1881
Unknown            1570
formerly smoked     868
smokes              791
Name: count, dtype: int64


**Tres observaciones que condicionan todo lo que viene.**

1. **`bmi` tiene nulos reales.** Hay que decidir qué hacer con ellos y justificarlo.
2. **`Unknown` en `smoking_status` no es un nulo.** Es una categoría declarada por la
   fuente: significa que no se registró el dato. Tratarla como faltante e imputarla
   sería inventar información sobre personas a las que nunca se les preguntó.
3. **La variable objetivo está muy desbalanceada**, cerca del 5 por ciento de casos
   positivos. Eso importa para las fases siguientes y conviene tenerlo declarado.

---
# 1. De funciones a objetos: por qué

Así se ve el código de la Fase 2, en celdas sueltas:

In [12]:
# El estilo de la Fase 2: funciona, pero cada celda depende de las anteriores
df_f2 = datos.copy()
df_f2 = df_f2.drop(columns=["id"])
df_f2["bmi"] = df_f2["bmi"].fillna(df_f2["bmi"].median())
df_f2 = pd.get_dummies(df_f2, columns=["work_type"], dtype=int)

print("Resultado:", df_f2.shape)

Resultado: (5110, 15)


Funciona. Pero tiene cuatro problemas que aparecen cuando el proyecto crece:

| Problema | Consecuencia |
|---|---|
| El orden importa y no está declarado | Ejecutar una celda fuera de orden produce otro resultado |
| No se puede reutilizar sin copiar | El mismo código se duplica en el cuaderno de F3 y F4 |
| No se puede probar por partes | Si el resultado está mal, hay que revisar todo |
| La mediana se calcula sobre todo el conjunto | Si después separan entrenamiento y prueba, hay **fuga de datos** |

La programación orientada a objetos resuelve los cuatro: cada paso pasa a ser una pieza
con nombre, con estado propio y con una interfaz fija.

---
# 2. Programación orientada a objetos desde cero

Esta parte es para quienes nunca han trabajado con clases. Si ya conocen el paradigma,
pueden pasar a la parte 3, pero conviene mirar el trazado de la celda 2.3: explica algo
que casi nadie enseña y que aclara `self` de una vez.

## 2.1 La idea: un molde y sus copias

Una **clase** es un molde. No es un dato: es la descripción de qué datos guarda algo y
qué operaciones sabe hacer.

Un **objeto** es una copia concreta hecha con ese molde. De un mismo molde se pueden
sacar muchas copias, y cada una guarda sus propios valores.

Una analogía que funciona: la clase es el formulario en blanco y el objeto es un
formulario llenado. El formulario en blanco define qué campos hay; cada formulario
llenado tiene sus propios valores en esos campos.

En ciencia de datos lo usamos cuando un componente necesita **recordar algo** entre una
llamada y otra. Ese es el criterio para decidir entre función y clase.

In [13]:
# Una clase mínima, con todo lo que hay que entender
class Contador:
    """Cuenta cuántas veces se le pide contar.

    El docstring, este texto entre comillas triples, es la documentación de la
    clase. Aparece al escribir help(Contador).
    """

    def __init__(self):
        # __init__ es el CONSTRUCTOR: Python lo ejecuta solo al crear el objeto.
        # Su trabajo es dejar listos los atributos.
        # self es el objeto que se está creando.
        self.total = 0          # ATRIBUTO: un dato que el objeto guarda

    def sumar(self, cantidad=1):
        # MÉTODO: una operación que el objeto sabe hacer.
        # Recibe self porque necesita saber sobre CUÁL objeto opera.
        self.total = self.total + cantidad
        return self.total


# Crear un objeto: acá Python ejecuta __init__ por nosotros
c = Contador()
print("Al crearlo, total vale:", c.total)

c.sumar()        # sin argumento: usa cantidad=1 por defecto
c.sumar(5)
print("Después de sumar 1 y 5:", c.total)
print("Tipo del objeto:", type(c))

Al crearlo, total vale: 0
Después de sumar 1 y 5: 6
Tipo del objeto: <class '__main__.Contador'>


## 2.2 Cada objeto tiene sus propios datos

Este es el punto que más cuesta al principio: dos objetos de la misma clase **no
comparten** sus atributos.

In [14]:
c1 = Contador()          # primer formulario llenado
c2 = Contador()          # segundo formulario, independiente del primero

c1.sumar(10)
c2.sumar(3)

print("c1.total:", c1.total)
print("c2.total:", c2.total)
print("¿Son el mismo objeto?", c1 is c2)
print("\nCada objeto guarda lo suyo. El molde es el mismo; los datos, no.")

# Lo que cada objeto guarda se puede ver directamente
print("\nContenido interno de c1:", c1.__dict__)
print("Contenido interno de c2:", c2.__dict__)

c1.total: 10
c2.total: 3
¿Son el mismo objeto? False

Cada objeto guarda lo suyo. El molde es el mismo; los datos, no.

Contenido interno de c1: {'total': 10}
Contenido interno de c2: {'total': 3}


## 2.3 Qué es `self`, explicado con el trazado real

`self` es la respuesta a una pregunta práctica: si el método está escrito una sola vez
en la clase, **¿cómo sabe sobre cuál objeto trabajar?**

La respuesta es que Python le pasa el objeto como primer argumento. Cuando ustedes
escriben `c1.sumar(10)`, Python ejecuta por detrás `Contador.sumar(c1, 10)`.

Eso significa que `self` no tiene nada de mágico: es simplemente el primer parámetro, y
podría llamarse de otra forma. Se llama `self` por convención, y conviene respetarla.

In [15]:
c3 = Contador()

# Las dos líneas siguientes hacen exactamente lo mismo
c3.sumar(7)                 # forma normal
Contador.sumar(c3, 7)       # lo que Python ejecuta por detrás

print("Total tras las dos llamadas equivalentes:", c3.total)
print("\nPor eso todo método lleva self como primer parámetro:")
print("es el espacio donde llega el objeto sobre el que se opera.")

Total tras las dos llamadas equivalentes: 14

Por eso todo método lleva self como primer parámetro:
es el espacio donde llega el objeto sobre el que se opera.


## 2.4 Dónde busca Python un atributo

Cuando escriben `objeto.algo`, Python lo busca en este orden:

1. En el **objeto**, es decir en su `__dict__`.
2. Si no está, en su **clase**.
3. Si no está, en la **clase madre**, y así hacia arriba.

Esa cadena se llama MRO, por *method resolution order*, y se puede consultar. Entenderla
explica por qué la herencia funciona.

In [16]:
class Persona:
    especie = "humano"                  # atributo de CLASE: uno solo, compartido

    def __init__(self, nombre):
        self.nombre = nombre            # atributo de INSTANCIA: uno por objeto


ana = Persona("Ana")
luis = Persona("Luis")

print("Atributos propios de ana :", ana.__dict__)
print("Atributo de la clase     :", Persona.especie)
print("ana.especie ->", ana.especie, "(no está en ana: Python sube a la clase)")

# Cambiar el atributo de clase afecta a TODOS los objetos
Persona.especie = "homo sapiens"
print("\nDespués de cambiarlo en la clase:")
print("  ana.especie :", ana.especie)
print("  luis.especie:", luis.especie)

# Cambiar un atributo de instancia afecta solo a ese objeto
ana.nombre = "Ana María"
print("\nDespués de cambiar solo ana.nombre:")
print("  ana.nombre :", ana.nombre)
print("  luis.nombre:", luis.nombre)

print("\nCadena de búsqueda (MRO):", [c.__name__ for c in Persona.__mro__])

Atributos propios de ana : {'nombre': 'Ana'}
Atributo de la clase     : humano
ana.especie -> humano (no está en ana: Python sube a la clase)

Después de cambiarlo en la clase:
  ana.especie : homo sapiens
  luis.especie: homo sapiens

Después de cambiar solo ana.nombre:
  ana.nombre : Ana María
  luis.nombre: Luis

Cadena de búsqueda (MRO): ['Persona', 'object']


**La conclusión práctica.** Un atributo de clase sirve para constantes compartidas, como
una etiqueta o un valor por defecto. Un atributo de instancia sirve para lo que cambia
de un objeto a otro, que es casi todo lo que nos interesa en un pipeline.

## 2.5 Resumen de los cinco conceptos

| Concepto | Qué es | Cómo se reconoce en el código |
|---|---|---|
| **Clase** | El molde | `class NombreDeLaClase:` |
| **Objeto** | Una copia del molde | `obj = NombreDeLaClase(...)` |
| **Atributo** | Un dato que el objeto guarda | `self.algo = valor` |
| **Método** | Una operación del objeto | `def hacer(self, ...):` |
| **`self`** | El objeto sobre el que se opera | Primer parámetro de todo método |

Con esos cinco alcanza para leer todo lo que sigue.

---
# 3. La clase `Preprocesador`

Primera versión: una clase que guarda la tabla y sabe prepararla. Cada método hace
**una sola cosa** y devuelve `self`, lo que permite encadenar llamadas.

In [17]:
class Preprocesador:
    """Guarda la tabla de stroke y sabe prepararla para el análisis.

    La tabla vive en self.df. Cada método es un paso del pipeline que trabaja
    sobre esa misma tabla y devuelve self, para poder encadenar.
    """

    def __init__(self, df):
        self.df = df.copy()          # copia: no se modifica el original
        self.registro = []           # lo que el objeto va recordando

    def quitar_identificador(self):
        """El id no aporta información al análisis: identifica, no describe."""
        if "id" in self.df.columns:
            self.df = self.df.drop(columns=["id"])
            self.registro.append("id eliminado")
        return self

    def imputar_bmi(self):
        """Rellena los nulos de bmi con la mediana."""
        nulos = int(self.df["bmi"].isna().sum())
        mediana = float(self.df["bmi"].median())
        self.df["bmi"] = self.df["bmi"].fillna(mediana)
        self.registro.append(f"bmi: {nulos} nulos imputados con mediana={mediana:.1f}")
        return self

    def codificar(self, columnas):
        """Convierte columnas de categorías en columnas 0/1."""
        antes = self.df.shape[1]
        self.df = pd.get_dummies(self.df, columns=columnas, dtype=int)
        self.registro.append(f"codificadas {columnas}: {antes} -> {self.df.shape[1]} columnas")
        return self

    def informe(self):
        """Devuelve lo que el objeto recuerda haber hecho."""
        return "\n".join(f"  {i}. {paso}" for i, paso in enumerate(self.registro, 1))


prep = Preprocesador(datos)
prep.quitar_identificador().imputar_bmi().codificar(["work_type", "smoking_status"])

print("Resultado:", prep.df.shape)
print("\nLo que el objeto recuerda:")
print(prep.informe())

Resultado: (5110, 18)

Lo que el objeto recuerda:
  1. id eliminado
  2. bmi: 201 nulos imputados con mediana=28.8
  3. codificadas ['work_type', 'smoking_status']: 11 -> 18 columnas


**Lo que ganamos con esto.** El objeto `prep` recuerda qué hizo y con qué valores. En la
Fase 2 esa información había que anotarla a mano en la bitácora; acá es producto de la
ejecución y no puede quedar desactualizada.

**Lo que todavía falta.** Esta clase mezcla dos cosas: aprender de los datos y
aplicarlos. Eso se arregla en la parte siguiente.

---
# 4. Encapsulamiento: proteger el estado interno

El `Preprocesador` de arriba tiene un problema serio: calcula la mediana **sobre el
conjunto completo**. Si después separan entrenamiento y prueba, la mediana ya vio los
datos de prueba. Eso se llama **fuga de datos** y es uno de los errores más caros en
ciencia de datos.

La solución es separar dos momentos:

- **`ajustar`**: aprende los parámetros, y solo del conjunto de entrenamiento.
- **`transformar`**: los aplica a cualquier conjunto.

Y proteger ese estado interno con un guion bajo al inicio del nombre, que en Python
significa «esto es interno, no lo toques desde fuera».

In [ ]:
class ImputadorBMI:
    """Imputa bmi separando lo que aprende de lo que aplica."""

    def __init__(self):
        self._mediana = None        # interno: se aprende, no se asigna desde fuera
        self._ajustado = False      # interno: controla el orden de las llamadas

    @property
    def ajustado(self):
        """Solo lectura: se puede consultar, no asignar."""
        return self._ajustado

    @property
    def mediana(self):
        return self._mediana

    def ajustar(self, df):
        self._mediana = float(df["bmi"].median())
        self._ajustado = True
        return self

    def transformar(self, df):
        if not self._ajustado:
            raise RuntimeError(
                "ImputadorBMI: hay que llamar a ajustar() antes de transformar(). "
                "La mediana se aprende del conjunto de entrenamiento."
            )
        df = df.copy()
        df["bmi"] = df["bmi"].fillna(self._mediana)
        return df


# El orden equivocado ahora falla con un mensaje claro
imputador = ImputadorBMI()
try:
    imputador.transformar(datos)
except RuntimeError as error:
    print("RuntimeError:", error)

RuntimeError: ImputadorBMI: hay que llamar a ajustar() antes de transformar(). La mediana se aprende del conjunto de entrenamiento.


In [ ]:
# El orden correcto, con separación de conjuntos
entrenamiento = datos.sample(frac=1 - PROPORCION_PRUEBA, random_state=SEMILLA)
prueba = datos.drop(index=entrenamiento.index)

imputador = ImputadorBMI().ajustar(entrenamiento)
prueba_lista = imputador.transformar(prueba)

print("Mediana aprendida del entrenamiento:", round(imputador.mediana, 2))
print("Mediana del conjunto de prueba     :", round(prueba["bmi"].median(), 2))
print("Nulos en prueba después de imputar :", int(prueba_lista["bmi"].isna().sum()))
print("\nSon medianas distintas, y está bien: se usó la del entrenamiento.")

# La propiedad es de solo lectura
try:
    imputador.ajustado = False
except AttributeError as error:
    print("\nAttributeError:", error)

Mediana aprendida del entrenamiento: 28.7
Mediana del conjunto de prueba     : 29.1
Nulos en prueba después de imputar : 0

Son medianas distintas, y está bien: se usó la del entrenamiento.

AttributeError: property 'ajustado' of 'ImputadorBMI' object has no setter


**Por qué importa.** Sin el control de `_ajustado`, transformar antes de ajustar
produciría un resultado sin sentido **y sin ningún mensaje de error**. El
encapsulamiento convierte un error silencioso en uno visible.

En el informe, este bloque es la evidencia del encapsulamiento: atributos internos con
guion bajo, propiedades de solo lectura y validación antes de actuar.

---
# 5. Herencia y polimorfismo

Miren el `ImputadorBMI`: el control de `_ajustado`, la copia defensiva y el mensaje de
error van a repetirse en el codificador, en el escalador y en cualquier otro paso.

**Herencia** es escribir eso una sola vez en una clase base.

In [ ]:
class Transformador:
    """Clase base: lo que todos los pasos del pipeline comparten.

    Esta clase NO se usa directamente. Su trabajo es definir el contrato:
    qué métodos tendrá todo paso del pipeline y cómo se controla su estado.
    Las clases hijas solo completan lo que cambia de un paso a otro.
    """

    def __init__(self, columna):
        # Atributo público: cualquiera puede leerlo y cambiarlo
        self.columna = columna

        # Atributos con guion bajo: por convención, INTERNOS.
        # Python no lo impide, pero el guion bajo le dice a quien lee el código
        # que no debe tocarlos desde fuera de la clase.
        self._parametros = {}       # lo que el paso aprende del conjunto
        self._ajustado = False      # controla que no se transforme antes de ajustar

    # @property convierte un método en algo que se lee como atributo:
    # se escribe paso.nombre y no paso.nombre(). Y como no hay setter,
    # el valor NO se puede asignar desde fuera. Eso es encapsulamiento.
    @property
    def nombre(self):
        # type(self).__name__ devuelve el nombre de la clase REAL del objeto,
        # así que un ImputadorMediana se identifica como tal y no como Transformador
        return f"{type(self).__name__}({self.columna})"

    @property
    def parametros(self):
        # dict(...) crea una COPIA. Si devolviéramos self._parametros
        # directamente, quien lo recibe podría modificar el estado interno
        # del objeto sin que la clase se entere. A esto se le llama
        # copia defensiva.
        return dict(self._parametros)

    def ajustar(self, df):
        """Aprende los parámetros del conjunto que recibe."""
        # 1) Validar la entrada ANTES de trabajar: si algo falta, se avisa
        #    acá y no veinte líneas más abajo con un error incomprensible
        if self.columna not in df.columns:
            raise KeyError(f"{self.nombre}: la columna no existe en el conjunto.")

        # 2) Delegar el cálculo a la clase hija. La base no sabe QUÉ se aprende;
        #    solo sabe CUÁNDO hay que aprenderlo.
        self._parametros = self.aprender(df)

        # 3) Registrar que ya se ajustó, para poder controlarlo después
        self._ajustado = True

        # 4) Devolver self permite encadenar: paso.ajustar(df).transformar(df)
        return self

    def transformar(self, df):
        """Aplica la transformación usando lo aprendido en ajustar()."""
        # Esta guarda es el corazón del encapsulamiento: impide un uso
        # incorrecto que de otro modo pasaría inadvertido
        if not self._ajustado:
            raise RuntimeError(f"{self.nombre}: hay que ajustar antes de transformar.")

        # df.copy() evita modificar el DataFrame que nos pasaron. Sin esto,
        # el objeto original del usuario cambiaría sin que él lo pidiera.
        return self.aplicar(df.copy())

    def ajustar_transformar(self, df):
        """Atajo para el caso habitual: aprender y aplicar sobre el mismo conjunto."""
        return self.ajustar(df).transformar(df)

    # ---- Métodos que cada clase hija DEBE implementar -------------------
    # Se definen acá lanzando NotImplementedError para dejar el contrato
    # explícito: si una hija olvida implementarlos, el error lo dice claro.

    def aprender(self, df):
        """Calcula y devuelve los parámetros. Lo implementa cada clase hija."""
        raise NotImplementedError("Cada clase hija debe implementar aprender().")

    def aplicar(self, df):
        """Aplica la transformación. Lo implementa cada clase hija."""
        raise NotImplementedError("Cada clase hija debe implementar aplicar().")


# La clase base no se usa sola
try:
    Transformador("bmi").ajustar(datos)
except NotImplementedError as error:
    print("NotImplementedError:", error)

NotImplementedError: Cada clase hija debe implementar aprender().


In [ ]:
class ImputadorMediana(Transformador):
    """Rellena los nulos con la mediana aprendida.

    El paréntesis (Transformador) es la HERENCIA: esta clase recibe todo lo
    que tiene Transformador sin volver a escribirlo. Solo implementa los dos
    métodos que le faltaban.
    """

    def aprender(self, df):
        return {"mediana": float(df[self.columna].median()),
                "nulos_en_ajuste": int(df[self.columna].isna().sum())}

    def aplicar(self, df):
        df[self.columna] = df[self.columna].fillna(self._parametros["mediana"])
        return df


class CodificadorNominal(Transformador):
    """Convierte una columna de categorías en columnas 0/1.

    El vocabulario se aprende en el ajuste: una categoría que solo aparece en
    prueba no genera columna nueva, porque el modelo no pudo aprender de ella.
    """

    def aprender(self, df):
        # Se guarda la lista de categorías ORDENADA, para que el resultado sea
        # el mismo en cada ejecución. Sin sorted(), el orden podría variar.
        return {"categorias": sorted(df[self.columna].dropna().unique())}

    def aplicar(self, df):
        # Se recorre el vocabulario aprendido, NO las categorías de este df.
        # Por eso una categoría nueva en prueba no genera columna: el modelo
        # nunca la vio y no podría haber aprendido nada de ella.
        for categoria in self._parametros["categorias"]:
            # Los nombres de columna no deben llevar espacios ni guiones
            etiqueta = str(categoria).strip().replace(" ", "_").replace("-", "_")
            # La comparación devuelve True/False; astype(int) lo pasa a 1/0
            df[f"{self.columna}_{etiqueta}"] = (df[self.columna] == categoria).astype(int)

        # La columna original ya no aporta: su información quedó en las nuevas
        return df.drop(columns=[self.columna])


class EscaladorEstandar(Transformador):
    """Centra en cero y escala a desviación uno."""

    def aprender(self, df):
        desviacion = float(df[self.columna].std())
        return {"media": float(df[self.columna].mean()),
                "desviacion": desviacion if desviacion != 0 else 1.0}

    def aplicar(self, df):
        df[self.columna] = ((df[self.columna] - self._parametros["media"])
                            / self._parametros["desviacion"])
        return df


class EliminadorColumnas(Transformador):
    """Quita columnas que no aportan al análisis, como el identificador."""

    def aprender(self, df):
        return {"existe": self.columna in df.columns}

    def aplicar(self, df):
        return df.drop(columns=[self.columna]) if self.columna in df.columns else df


imp = ImputadorMediana("bmi")
salida = imp.ajustar_transformar(datos)
print(imp.nombre, "->", imp.parametros)
print("Nulos en bmi después:", int(salida["bmi"].isna().sum()))

ImputadorMediana(bmi) -> {'mediana': 28.8, 'nulos_en_ajuste': 201}
Nulos en bmi después: 0


### Cómo comprobar que la herencia funciona

No hay que creer en la herencia: se puede verificar.

In [ ]:
imputador = ImputadorMediana("bmi")

# 1) El objeto ES un ImputadorMediana Y TAMBIÉN es un Transformador
print("¿Es ImputadorMediana?", isinstance(imputador, ImputadorMediana))
print("¿Es Transformador?   ", isinstance(imputador, Transformador))

# 2) La cadena de búsqueda muestra de dónde hereda
print("\\nCadena de herencia:", [c.__name__ for c in ImputadorMediana.__mro__])

# 3) Métodos que NO están escritos en la clase hija y sin embargo funcionan,
#    porque se heredaron de la clase base
print("\\nMétodos heredados de Transformador:")
for metodo in ["ajustar", "transformar", "ajustar_transformar"]:
    definido_en = "ImputadorMediana" if metodo in ImputadorMediana.__dict__ else "Transformador"
    print(f"  {metodo:<22} definido en {definido_en}")

print("\\nMétodos propios de la clase hija:")
for metodo in ["aprender", "aplicar"]:
    print(f"  {metodo:<22} definido en ImputadorMediana")

¿Es ImputadorMediana? True
¿Es Transformador?    True
\nCadena de herencia: ['ImputadorMediana', 'Transformador', 'object']
\nMétodos heredados de Transformador:
  ajustar                definido en Transformador
  transformar            definido en Transformador
  ajustar_transformar    definido en Transformador
\nMétodos propios de la clase hija:
  aprender               definido en ImputadorMediana
  aplicar                definido en ImputadorMediana


**Lo que acaban de ver.** `ImputadorMediana` tiene ocho líneas de código propio y
dispone de toda la maquinaria de control de estado. Eso es lo que la herencia ahorra: si
mañana hay que cambiar el mensaje de error, se cambia en un solo lugar y las cuatro
clases hijas lo heredan.

**Polimorfismo** es que quien usa estas clases no necesita saber de qué tipo es cada
una: todas ofrecen `ajustar` y `transformar`.

In [ ]:
pasos = [
    EliminadorColumnas("id"),
    ImputadorMediana("bmi"),
    CodificadorNominal("work_type"),
    CodificadorNominal("smoking_status"),
    EscaladorEstandar("age"),
]

resultado = datos.copy()
for paso in pasos:                      # la misma llamada para los cinco
    resultado = paso.ajustar_transformar(resultado)
    print(f"{paso.nombre:<34} -> {resultado.shape}")

EliminadorColumnas(id)             -> (5110, 11)
ImputadorMediana(bmi)              -> (5110, 11)
CodificadorNominal(work_type)      -> (5110, 15)
CodificadorNominal(smoking_status) -> (5110, 18)
EscaladorEstandar(age)             -> (5110, 18)


**Este bloque es la evidencia de los tres principios**, y así conviene declararlo en el
informe:

- **Herencia**: las cuatro clases heredan de `Transformador` y no repiten el control de estado.
- **Polimorfismo**: el bucle llama `ajustar_transformar()` sin preguntar el tipo.
- **Encapsulamiento**: `_parametros` y `_ajustado` son internos, y hay validación antes de actuar.

Y fíjense en la consecuencia práctica: **agregar un transformador nuevo no obliga a
tocar el bucle**. Eso es lo que la rúbrica llama bajo acoplamiento.

---
# 6. El `Pipeline`: componer los pasos

Una clase que guarda el orden, los ejecuta y controla que se haya ajustado antes de
transformar.

In [ ]:
class Pipeline:
    """Encadena transformadores y los ejecuta en orden."""

    def __init__(self, pasos=None):
        self._pasos = list(pasos) if pasos else []
        self._ajustado = False

    def agregar(self, transformador):
        if not isinstance(transformador, Transformador):
            raise TypeError(
                f"Se esperaba un Transformador y se recibió {type(transformador).__name__}."
            )
        self._pasos.append(transformador)
        return self

    def ajustar(self, df):
        """Aprende los parámetros de cada paso SOLO con estos datos."""
        intermedio = df.copy()
        for paso in self._pasos:
            intermedio = paso.ajustar_transformar(intermedio)
        self._ajustado = True
        return self

    def transformar(self, df):
        if not self._ajustado:
            raise RuntimeError("Pipeline: hay que ajustar antes de transformar.")
        resultado = df.copy()
        for paso in self._pasos:
            resultado = paso.transformar(resultado)
        return resultado

    def pasos_ejecutados(self):
        """Devuelve los pasos, para poder registrar sus parámetros aprendidos."""
        return tuple(self._pasos)

    def resumen(self):
        return pd.DataFrame([
            {"orden": i, "paso": p.nombre, "clase": type(p).__name__}
            for i, p in enumerate(self._pasos, start=1)
        ])

    def __len__(self):
        return len(self._pasos)

    def __repr__(self):
        estado = "ajustado" if self._ajustado else "sin ajustar"
        return f"Pipeline({len(self._pasos)} pasos, {estado})"


pipeline = Pipeline([
    EliminadorColumnas("id"),
    ImputadorMediana("bmi"),
    CodificadorNominal("work_type"),
    CodificadorNominal("smoking_status"),
    CodificadorNominal("gender"),
    EscaladorEstandar("age"),
    EscaladorEstandar("avg_glucose_level"),
    EscaladorEstandar("bmi"),
])

print(pipeline)
pipeline.resumen()

Pipeline(8 pasos, sin ajustar)


,orden,paso,clase
0,1,EliminadorColumnas(id),EliminadorColumnas
1,2,ImputadorMediana(bmi),ImputadorMediana
2,3,CodificadorNominal(work_type),CodificadorNominal
3,4,CodificadorNominal(smoking_status),CodificadorNominal
4,5,CodificadorNominal(gender),CodificadorNominal
5,6,EscaladorEstandar(age),EscaladorEstandar
6,7,EscaladorEstandar(avg_glucose_level),EscaladorEstandar
7,8,EscaladorEstandar(bmi),EscaladorEstandar


In [ ]:
# Ajustar con entrenamiento, transformar prueba: sin fuga de datos
pipeline.ajustar(entrenamiento)
prueba_lista = pipeline.transformar(prueba)

print("Entrenamiento:", entrenamiento.shape, " -> Prueba transformada:", prueba_lista.shape)
print("\nMedia de age en prueba tras escalar:", round(prueba_lista["age"].mean(), 4))
print("No es exactamente cero, y está bien: los parámetros vienen del entrenamiento.")

Entrenamiento: (4088, 12)  -> Prueba transformada: (1022, 20)

Media de age en prueba tras escalar: -0.0078
No es exactamente cero, y está bien: los parámetros vienen del entrenamiento.


**Por qué la media no da cero.** El escalador aprendió la media del entrenamiento. Si
diera exactamente cero sobre prueba, significaría que aprendió de datos que no debía
ver. Este resultado es la confirmación de que el diseño está bien.

---
# 7. Cohesión y acoplamiento

Son las dos palabras que más aparecen en el criterio de diseño estructurado y que casi
ningún material explica. Acá van con código.

**Alta cohesión** significa que cada componente hace **una sola cosa**.

**Bajo acoplamiento** significa que los componentes dependen poco unos de otros: cambiar
uno no obliga a tocar los demás.

## 7.1 Cohesión baja: una clase que hace de todo

Este es el diseño que aparece cuando se escribe rápido. Funciona, pero cada método toca
cosas distintas y la clase no tiene un propósito único.

In [ ]:
class ProcesadorTodoEnUno:
    """Ejemplo de COHESIÓN BAJA: hace cuatro cosas sin relación entre sí."""

    def __init__(self, ruta):
        self.ruta = ruta
        self.df = None

    def cargar(self):            # responsabilidad 1: entrada y salida de archivos
        self.df = pd.read_csv(self.ruta)

    def limpiar(self):           # responsabilidad 2: transformar datos
        self.df = self.df.dropna()

    def graficar(self):          # responsabilidad 3: visualización
        pass

    def enviar_correo(self):     # responsabilidad 4: comunicación
        pass


print("Cuatro responsabilidades distintas en una sola clase.")
print("Problema práctico: para probar la limpieza hay que tener un archivo real,")
print("porque cargar() y limpiar() viven pegadas en el mismo objeto.")

Cuatro responsabilidades distintas en una sola clase.
Problema práctico: para probar la limpieza hay que tener un archivo real,
porque cargar() y limpiar() viven pegadas en el mismo objeto.


## 7.2 Cohesión alta: cada clase con un propósito

El mismo trabajo, repartido. Ahora cada pieza se puede probar sola.

In [ ]:
class Cargador:
    """Una sola responsabilidad: leer datos desde una fuente."""

    def __init__(self, ruta):
        self.ruta = ruta

    def cargar(self):
        return pd.read_csv(self.ruta)


class Limpiador:
    """Una sola responsabilidad: transformar un DataFrame que le entregan.

    Fíjense en que NO sabe de dónde vienen los datos. Recibe un DataFrame y
    devuelve otro. Eso es bajo acoplamiento: no depende del Cargador.
    """

    def limpiar(self, df):
        return df.dropna()


# La ventaja se ve al probar: no hace falta ningún archivo
mini = pd.DataFrame({"a": [1.0, np.nan, 3.0], "b": [4.0, 5.0, 6.0]})
print("Se puede probar el Limpiador sin tocar el disco:")
print("  entrada:", mini.shape, " -> salida:", Limpiador().limpiar(mini).shape)

Se puede probar el Limpiador sin tocar el disco:
  entrada: (3, 2)  -> salida: (2, 2)


## 7.3 La prueba del acoplamiento

Hay una pregunta que resuelve el asunto sin teoría:

> **Si cambiamos la forma de imputar, ¿cuántos archivos hay que modificar?**

Si la respuesta es **uno**, el acoplamiento es bajo. Si son tres, el diseño es frágil.

Veámoslo con el pipeline que ya construimos.

In [ ]:
# Cambiar un paso del pipeline no obliga a tocar ni el Pipeline ni los otros pasos
pipeline_a = Pipeline([EliminadorColumnas("id"), ImputadorMediana("bmi")])
pipeline_b = Pipeline([EliminadorColumnas("id"), ImputadorMediana("avg_glucose_level")])

for nombre, pipe in [("versión A", pipeline_a), ("versión B", pipeline_b)]:
    salida = pipe.ajustar(datos).transformar(datos)
    print(f"{nombre}: {salida.shape}")

print("\nSe cambió el paso y NO se modificó la clase Pipeline ni las demás clases.")
print("Eso es bajo acoplamiento, y es lo que el criterio de diseño evalúa.")

versión A: (5110, 11)
versión B: (5110, 11)

Se cambió el paso y NO se modificó la clase Pipeline ni las demás clases.
Eso es bajo acoplamiento, y es lo que el criterio de diseño evalúa.


## 7.4 Cómo se traduce en la estructura de archivos

Una organización con alta cohesión y bajo acoplamiento se reconoce a simple vista:

```
src/
├── carga.py            solo lee y verifica archivos
├── transformadores.py  solo las clases de transformación
├── pipeline.py         solo encadena y ejecuta
└── medicion.py         solo compara implementaciones
```

Cada archivo tiene un propósito que se puede enunciar en una frase. Si al describir un
archivo hace falta la palabra «y» varias veces, probablemente tenga cohesión baja.

**Para el informe.** El apartado de diseño estructurado pide justificar la organización.
La prueba del acoplamiento es el mejor argumento: «para cambiar la estrategia de
imputación solo se modifica `transformadores.py`, porque el pipeline no conoce el
detalle de cada paso».

---
# 8. Validación: casos normales, límite y excepciones

La rúbrica pide los tres escenarios. Acá va uno de cada tipo, con el conjunto de stroke.

In [ ]:
# --- CASO NORMAL: el flujo completo sobre el conjunto real ---
pipe = Pipeline([EliminadorColumnas("id"), ImputadorMediana("bmi")])
salida = pipe.ajustar(datos).transformar(datos)

assert len(salida) == len(datos), "El pipeline perdió o duplicó filas"
assert salida["bmi"].isna().sum() == 0, "Quedaron nulos en bmi"
assert "id" not in salida.columns, "La columna id no se eliminó"
print("Caso normal: las tres comprobaciones pasaron.")

Caso normal: las tres comprobaciones pasaron.


In [ ]:
# --- CASOS LÍMITE: situaciones extremas pero válidas ---

# 1) Una columna sin ningún nulo no debe alterarse
sin_nulos = pd.DataFrame({"bmi": [25.0, 30.0, 35.0]})
r1 = ImputadorMediana("bmi").ajustar_transformar(sin_nulos)
assert r1["bmi"].tolist() == [25.0, 30.0, 35.0]
print("Límite 1: columna sin nulos, sin cambios.")

# 2) Una sola categoría genera una sola columna
una_cat = pd.DataFrame({"work_type": ["Private", "Private", "Private"]})
r2 = CodificadorNominal("work_type").ajustar_transformar(una_cat)
print("Límite 2: una categoría ->", list(r2.columns))

# 3) Varianza cero no debe producir división por cero
constante = pd.DataFrame({"age": [50.0, 50.0, 50.0]})
r3 = EscaladorEstandar("age").ajustar_transformar(constante)
assert r3["age"].notna().all()
print("Límite 3: varianza cero ->", r3["age"].tolist())

# 4) Una categoría nueva en prueba no debe crear columna
pipe_cat = Pipeline([CodificadorNominal("gender")]).ajustar(entrenamiento)
nuevos = prueba.copy()
nuevos.iloc[0, nuevos.columns.get_loc("gender")] = "No informado"
r4 = pipe_cat.transformar(nuevos)
print("Límite 4: categoría nueva no crea columna ->",
      "gender_No_informado" not in r4.columns)

Límite 1: columna sin nulos, sin cambios.
Límite 2: una categoría -> ['work_type_Private']
Límite 3: varianza cero -> [0.0, 0.0, 0.0]
Límite 4: categoría nueva no crea columna -> True


In [ ]:
# --- EXCEPCIONES: entradas que deben fallar con mensaje claro ---

pruebas = [
    ("transformar sin ajustar", lambda: ImputadorMediana("bmi").transformar(datos)),
    ("columna inexistente", lambda: ImputadorMediana("no_existe").ajustar(datos)),
    ("agregar algo que no es Transformador", lambda: Pipeline().agregar("texto")),
    ("pipeline sin ajustar", lambda: Pipeline([ImputadorMediana("bmi")]).transformar(datos)),
]

for descripcion, accion in pruebas:
    try:
        accion()
        print(f"  {descripcion:<38} NO lanzó excepción (revisar)")
    except (RuntimeError, KeyError, TypeError) as error:
        print(f"  {descripcion:<38} {type(error).__name__} capturado")

  transformar sin ajustar                RuntimeError capturado
  columna inexistente                    KeyError capturado
  agregar algo que no es Transformador   TypeError capturado
  pipeline sin ajustar                   RuntimeError capturado


**Su turno.** Repliquen esta estructura con su propio conjunto. Tres bloques: normal,
límite y excepción. Es evidencia directa del criterio de validación técnica, que vale
seis puntos.

## La verificación que más rinde

Si el pipeline con clases produce **el mismo resultado** que el código de la Fase 2,
tienen una prueba objetiva de que la reorganización no rompió nada.

In [ ]:
# Versión Fase 2: celdas sueltas
v_f2 = datos.copy()
v_f2 = v_f2.drop(columns=["id"])
v_f2["bmi"] = v_f2["bmi"].fillna(datos["bmi"].median())
for categoria in sorted(datos["work_type"].dropna().unique()):
    etiqueta = str(categoria).strip().replace(" ", "_").replace("-", "_")
    v_f2[f"work_type_{etiqueta}"] = (v_f2["work_type"] == categoria).astype(int)
v_f2 = v_f2.drop(columns=["work_type"])

# Versión Fase 3: el mismo trabajo con clases
v_f3 = Pipeline([
    EliminadorColumnas("id"),
    ImputadorMediana("bmi"),
    CodificadorNominal("work_type"),
]).ajustar(datos).transformar(datos)

pd.testing.assert_frame_equal(v_f2, v_f3)
print("Las dos versiones producen exactamente el mismo resultado.")
print("La reorganización no alteró nada.")

Las dos versiones producen exactamente el mismo resultado.
La reorganización no alteró nada.


---
# 9. Recursividad

Toda función recursiva tiene dos partes: un **caso base** que la detiene y un **caso
recursivo** que reduce el problema y vuelve a llamarse.

**Cuándo se justifica:** cuando no se sabe de antemano cuán profundo es el problema.
Si la profundidad es fija, un bucle es más simple y más rápido.

In [ ]:
def cuenta_regresiva(n):
    """Ejemplo mínimo para ver las dos partes."""
    if n == 0:                       # CASO BASE: detiene la recursión
        print("Despegue")
        return
    print(n, end=" ")
    cuenta_regresiva(n - 1)          # CASO RECURSIVO


cuenta_regresiva(5)

5

 4 3 2 1 Despegue


In [ ]:
def aplanar(estructura, prefijo=""):
    """Convierte metadatos anidados en pares plano de clave y valor.

    Por qué recursión y no un bucle: la profundidad no se conoce al escribir el
    código. Mañana alguien agrega un nivel y el bucle anidado deja de servir.
    """
    plano = {}
    for clave in estructura:
        valor = estructura[clave]
        compuesta = f"{prefijo}.{clave}" if prefijo else str(clave)
        if isinstance(valor, dict) and valor:
            plano.update(aplanar(valor, compuesta))
        else:
            plano[compuesta] = valor
    return plano


metadatos = {
    "proyecto": {"nombre": "stroke", "fase": 3},
    "datos": {"filas": len(datos), "nulos": {"bmi": int(datos["bmi"].isna().sum())}},
    "entorno": {"semilla": SEMILLA, "librerias": {"pandas": pd.__version__}},
}

for clave, valor in aplanar(metadatos).items():
    print(f"{clave:<26} {valor}")

proyecto.nombre            stroke
proyecto.fase              3
datos.filas                5110
datos.nulos.bmi            201
entorno.semilla            42
entorno.librerias.pandas   3.0.2


## El costo de repetir trabajo

La recursión **no es** sinónimo de eficiencia. Estas dos funciones tienen la misma
idea y un costo muy distinto.

In [ ]:
def fib_ingenua(n):
    """Sin memoria: recalcula los mismos subproblemas miles de veces."""
    if n < 2:
        return n
    return fib_ingenua(n - 1) + fib_ingenua(n - 2)


def fib_memoizada(n, cache=None):
    """La misma recursión, guardando lo ya calculado."""
    if cache is None:
        cache = {}
    if n in cache:
        return cache[n]
    resultado = n if n < 2 else fib_memoizada(n - 1, cache) + fib_memoizada(n - 2, cache)
    cache[n] = resultado
    return resultado


filas = []
for n in [18, 22, 26]:
    inicio = time.perf_counter(); fib_ingenua(n); t1 = time.perf_counter() - inicio
    inicio = time.perf_counter(); fib_memoizada(n); t2 = time.perf_counter() - inicio
    filas.append({"n": n, "ingenua_s": round(t1, 6), "memoizada_s": round(t2, 6),
                  "veces_mas_rapida": round(t1 / t2, 1)})

pd.DataFrame(filas)

,n,ingenua_s,memoizada_s,veces_mas_rapida
0,18,0.000257,0.000006,44.1
1,22,0.001759,0.000006,270.8
2,26,0.014983,0.000009,1660.0


El cambio entre una versión y otra es de dos líneas, y el efecto se mide en cientos de
veces. Lo que hace eficiente a la segunda no es la recursión: es **no repetir trabajo**.

**Si su proyecto no tiene un problema recursivo natural**, decláren­lo en el informe y
expliquen por qué un enfoque iterativo es preferible. La rúbrica acepta recursividad
**o** división funcional, y una decisión argumentada vale lo mismo que implementarla.

---
# 10. Eficiencia: medir tiempo y memoria

La rúbrica pide mediciones **reproducibles** con `timeit` o equivalente, comparación
entre implementaciones e interpretación considerando tiempo **y** memoria.

In [ ]:
def con_bucle(df):
    """Clasifica el IMC recorriendo las filas una por una."""
    categorias = []
    for valor in df["bmi"]:
        if pd.isna(valor):
            categorias.append("sin dato")
        elif valor < 18.5:
            categorias.append("bajo")
        elif valor < 25:
            categorias.append("normal")
        elif valor < 30:
            categorias.append("sobrepeso")
        else:
            categorias.append("obesidad")
    return categorias


def vectorizada(df):
    """Lo mismo, con una operación sobre la columna completa."""
    return pd.cut(df["bmi"], bins=[-np.inf, 18.5, 25, 30, np.inf],
                  labels=["bajo", "normal", "sobrepeso", "obesidad"],
                  right=False).astype(object).fillna("sin dato").tolist()


# Antes de comparar: comprobar que dan el mismo resultado
assert con_bucle(datos) == vectorizada(datos), "Las versiones no coinciden"
print("Las dos implementaciones producen el mismo resultado.\n")

t_bucle = timeit.timeit(lambda: con_bucle(datos), number=20)
t_vect = timeit.timeit(lambda: vectorizada(datos), number=20)

print(f"Con bucle   : {t_bucle:.4f} s")
print(f"Vectorizada : {t_vect:.4f} s")
print(f"La vectorizada es {t_bucle / t_vect:.1f} veces más rápida")

Las dos implementaciones producen el mismo resultado.



Con bucle   : 0.0216 s
Vectorizada : 0.0190 s
La vectorizada es 1.1 veces más rápida


**La comprobación con `assert` es obligatoria.** Una versión más rápida que entrega otro
resultado no es una optimización: es un error. Decláren­la en el informe.

In [ ]:
def medir(funcion, *args, **kwargs):
    """Ejecuta la función y devuelve resultado, segundos y memoria pico en MB.

    *args recoge los argumentos posicionales en una tupla y **kwargs los
    argumentos con nombre en un diccionario. El asterisco es lo que hace el
    trabajo; los nombres args y kwargs son solo convención.
    """
    tracemalloc.start()
    inicio = time.perf_counter()
    resultado = funcion(*args, **kwargs)
    transcurrido = time.perf_counter() - inicio
    _, pico = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return resultado, transcurrido, pico / 1024 / 1024


_, s_bucle, m_bucle = medir(con_bucle, datos)
_, s_vect, m_vect = medir(vectorizada, datos)

pd.DataFrame([
    {"version": "con bucle", "segundos": round(s_bucle, 5), "memoria_mb": round(m_bucle, 3)},
    {"version": "vectorizada", "segundos": round(s_vect, 5), "memoria_mb": round(m_vect, 3)},
])

,version,segundos,memoria_mb
0,con bucle,0.00432,0.041
1,vectorizada,0.00276,0.111


**Atención al contraste.** A veces la versión más rápida consume **más** memoria. Esa
tensión entre tiempo y espacio es exactamente lo que el descriptor pide analizar.

In [ ]:
# Cómo crece el costo con el tamaño: esto es lo que revela la complejidad
mediciones = []
for n in [1000, 5000, 20000, 50000]:
    muestra = datos.sample(n=n, replace=True, random_state=SEMILLA)
    t_b = timeit.timeit(lambda: con_bucle(muestra), number=5) / 5
    t_v = timeit.timeit(lambda: vectorizada(muestra), number=5) / 5
    mediciones.append({"filas": n, "bucle_ms": round(t_b * 1000, 2),
                       "vectorizada_ms": round(t_v * 1000, 2),
                       "razon": round(t_b / t_v, 1)})

tabla = pd.DataFrame(mediciones)
print(tabla.to_string(index=False))
print("\nLa última columna es lo que importa: la ventaja crece con el tamaño.")

 filas  bucle_ms  vectorizada_ms  razon
  1000      0.23            0.66    0.4
  5000      1.16            0.96    1.2
 20000      5.00            3.49    1.4
 50000     10.38            3.97    2.6

La última columna es lo que importa: la ventaja crece con el tamaño.


**Un resultado que conviene mirar con atención.** En los tamaños más pequeños la
versión vectorizada puede resultar **igual o más lenta**. No es un error: `pd.cut`
tiene un costo fijo de preparación que en mil filas pesa más que el ahorro. La ventaja
aparece cuando el volumen crece y ese costo fijo se reparte entre más datos.

Es exactamente el tipo de observación que la rúbrica premia: no basta con reportar que
una versión ganó, hay que explicar **bajo qué condiciones**.

**Tres reglas para que la medición valga**

1. **Repitan y conserven el tiempo menor**, no el promedio: los valores altos suelen
   reflejar interrupciones del sistema operativo, no el costo del código.
2. **Midan sobre el tamaño real** de su conjunto. En mil filas todo es instantáneo.
3. **Comprueben que las versiones dan el mismo resultado.**

**Lo que distingue el nivel superior:** que la medición **cambie una decisión**. Medir y
no cambiar nada es un ejercicio; medir, ver dónde está el costo y reescribir esa parte
es optimización.

---
# 11. Patrones de diseño

Un patrón es una solución conocida a un problema que se repite. Lo importante:
**ustedes ya usaron varios en la Fase 2 sin saber su nombre.**

## Strategy: varias formas de hacer lo mismo

En la Fase 2 compararon imputar `bmi` con la mediana, con la media o por grupo. Si cada
forma es un bloque distinto, cambiar de una a otra obliga a reescribir. Strategy separa
el **qué** del **cómo**.

In [ ]:
class EstrategiaImputacion:
    """Contrato común de todas las estrategias."""
    etiqueta = "sin definir"

    def calcular(self, df, columna):
        raise NotImplementedError


class PorMediana(EstrategiaImputacion):
    etiqueta = "mediana"

    def calcular(self, df, columna):
        return float(df[columna].median())


class PorMedia(EstrategiaImputacion):
    etiqueta = "media"

    def calcular(self, df, columna):
        return float(df[columna].mean())


class PorMedianaDeGrupo(EstrategiaImputacion):
    etiqueta = "mediana por grupo"

    def __init__(self, columna_grupo):
        self.columna_grupo = columna_grupo

    def calcular(self, df, columna):
        return df.groupby(self.columna_grupo)[columna].median().to_dict()


class ImputadorFlexible(Transformador):
    """No sabe imputar: sabe cuándo. El cómo lo aporta la estrategia."""

    def __init__(self, columna, estrategia=None):
        super().__init__(columna)          # llama al constructor de la clase base
        self.estrategia = estrategia or PorMediana()

    def aprender(self, df):
        return {"valor": self.estrategia.calcular(df, self.columna),
                "estrategia": self.estrategia.etiqueta}

    def aplicar(self, df):
        valor = self._parametros["valor"]
        if isinstance(valor, dict):                       # mediana por grupo
            relleno = df[self.estrategia.columna_grupo].map(valor)
            df[self.columna] = df[self.columna].fillna(relleno)
            df[self.columna] = df[self.columna].fillna(df[self.columna].median())
        else:
            df[self.columna] = df[self.columna].fillna(valor)
        return df


# La comparación de estrategias que pide la rúbrica, en un bucle
desv_original = datos["bmi"].std()
comparacion = []
for estrategia in [PorMediana(), PorMedia(), PorMedianaDeGrupo("gender")]:
    salida = ImputadorFlexible("bmi", estrategia).ajustar_transformar(datos)
    comparacion.append({
        "estrategia": estrategia.etiqueta,
        "desv_antes": round(desv_original, 4),
        "desv_despues": round(salida["bmi"].std(), 4),
        "cambio_pct": round((salida["bmi"].std() - desv_original) / desv_original * 100, 2),
    })

pd.DataFrame(comparacion)

,estrategia,desv_antes,desv_despues,cambio_pct
0,mediana,7.8398,7.6841,-1.99
1,media,7.8398,7.6841,-1.99
2,mediana por grupo,7.8398,7.6842,-1.99


**Nota sobre estos números.** Si el cuaderno está corriendo con datos de prueba, las
tres estrategias dan resultados casi idénticos porque el `bmi` simulado es simétrico.
Con el archivo real de stroke las diferencias son mayores, y ahí la comparación sí
discrimina. Ejecútenlo con su propio conjunto para ver la diferencia que importa.

Esa tabla es la comparación que el criterio de preprocesamiento pide justificar. Toda
imputación por un valor central reduce la dispersión; lo que se compara es **cuánto**, y
esa cifra es el argumento para elegir.

## Factory: decidir qué construir

La transformación que corresponde depende del rol de la variable. Factory concentra esa
decisión en un solo lugar, en vez de repetirla en cada cuaderno.

In [ ]:
def crear_transformador(rol, columna, **parametros):
    """Devuelve el transformador que corresponde al rol analítico."""
    rol = rol.strip().lower()

    if rol == "continua":
        return ImputadorFlexible(columna, parametros.get("estrategia"))
    if rol == "nominal":
        return CodificadorNominal(columna)
    if rol == "escalar":
        return EscaladorEstandar(columna)
    if rol == "identificador":
        return EliminadorColumnas(columna)

    raise ValueError(
        f"Rol desconocido: '{rol}'. "
        "Roles válidos: continua, nominal, escalar, identificador."
    )


# El diccionario de variables de la Fase 1 pasa a gobernar el pipeline
diccionario = [
    {"variable": "id", "rol": "identificador"},
    {"variable": "bmi", "rol": "continua"},
    {"variable": "work_type", "rol": "nominal"},
    {"variable": "smoking_status", "rol": "nominal"},
    {"variable": "age", "rol": "escalar"},
]

pipeline_auto = Pipeline([crear_transformador(v["rol"], v["variable"]) for v in diccionario])
salida = pipeline_auto.ajustar(datos).transformar(datos)

print(pipeline_auto.resumen().to_string(index=False))
print("\nResultado:", salida.shape)

try:
    crear_transformador("geografica", "Residence_type")
except ValueError as error:
    print("\nValueError:", error)

 orden                               paso              clase
     1             EliminadorColumnas(id) EliminadorColumnas
     2             ImputadorFlexible(bmi)  ImputadorFlexible
     3      CodificadorNominal(work_type) CodificadorNominal
     4 CodificadorNominal(smoking_status) CodificadorNominal
     5             EscaladorEstandar(age)  EscaladorEstandar

Resultado: (5110, 18)

ValueError: Rol desconocido: 'geografica'. Roles válidos: continua, nominal, escalar, identificador.


## Observer: registrar sin estorbar

La bitácora de decisiones suele escribirse a mano después de ejecutar, con el riesgo de
quedar desactualizada. Con Observer, el pipeline **avisa** cada vez que termina un paso
y quien quiera registrar se suscribe.

In [ ]:
class Bitacora:
    """Observador que acumula lo que ocurre y lo entrega como tabla."""

    def __init__(self):
        self.registros = []

    def notificar(self, paso, filas, columnas, segundos, parametros):
        self.registros.append({"paso": paso, "filas": filas, "columnas": columnas,
                               "segundos": round(segundos, 5), "parametros": parametros})

    def a_dataframe(self):
        return pd.DataFrame(self.registros)


class ReporteConsola:
    """Otro observador: imprime mientras ocurre."""

    def notificar(self, paso, filas, columnas, segundos, parametros):
        print(f"  {paso:<34} {filas:>6} filas, {columnas:>3} col  [{segundos:.4f}s]")


class PipelineObservable(Pipeline):
    """Hereda todo de Pipeline y agrega la capacidad de ser observado."""

    def __init__(self, pasos=None):
        super().__init__(pasos)
        self._observadores = []

    def suscribir(self, observador):
        self._observadores.append(observador)
        return self

    def transformar(self, df):
        if not self._ajustado:
            raise RuntimeError("Pipeline: hay que ajustar antes de transformar.")
        resultado = df.copy()
        for paso in self._pasos:
            inicio = time.perf_counter()
            resultado = paso.transformar(resultado)
            transcurrido = time.perf_counter() - inicio
            for observador in self._observadores:
                observador.notificar(paso.nombre, len(resultado), resultado.shape[1],
                                     transcurrido, paso.parametros)
        return resultado


bitacora = Bitacora()
pipe = PipelineObservable([
    EliminadorColumnas("id"),
    ImputadorFlexible("bmi"),
    CodificadorNominal("work_type"),
    EscaladorEstandar("age"),
])
pipe.suscribir(bitacora).suscribir(ReporteConsola())

print("Ejecución del pipeline:")
salida = pipe.ajustar(datos).transformar(datos)

print("\nBitácora producida por la propia ejecución:")
bitacora.a_dataframe()[["paso", "filas", "columnas", "segundos"]]

Ejecución del pipeline:
  EliminadorColumnas(id)               5110 filas,  11 col  [0.0005s]
  ImputadorFlexible(bmi)               5110 filas,  11 col  [0.0004s]
  CodificadorNominal(work_type)        5110 filas,  15 col  [0.0035s]
  EscaladorEstandar(age)               5110 filas,  15 col  [0.0007s]

Bitácora producida por la propia ejecución:


,paso,filas,columnas,segundos
0,EliminadorColumnas(id),5110,11,0.00053
1,ImputadorFlexible(bmi),5110,11,0.00043
2,CodificadorNominal(work_type),5110,15,0.00351
3,EscaladorEstandar(age),5110,15,0.00067


**Fíjense en lo que acaba de pasar.** La bitácora no se escribió a mano: se genera cada
vez que el pipeline corre, así que no puede quedar desactualizada.

## Singleton: una sola configuración

Rutas, semilla y umbrales se necesitan en varios módulos. Si cada uno construye su
propia configuración, dos partes del pipeline pueden usar semillas distintas sin que
nadie lo note.

**La advertencia:** Singleton introduce estado global, que complica las pruebas. Úsenlo
solo para configuración y pasen el resto de los parámetros explícitamente.

In [ ]:
class Configuracion:
    """Una sola instancia compartida por todo el proyecto."""

    _instancia = None

    def __new__(cls, *args, **kwargs):
        if cls._instancia is None:
            cls._instancia = super().__new__(cls)
            cls._instancia._iniciada = False
        return cls._instancia

    def __init__(self, semilla=SEMILLA):
        if self._iniciada:               # evita reiniciar una instancia existente
            return
        self.semilla = semilla
        self.parametros = {}
        self._iniciada = True

    def definir(self, clave, valor):
        self.parametros[clave] = valor
        return self


a = Configuracion(semilla=42)
b = Configuracion(semilla=99)            # se ignora: la instancia ya existe
a.definir("umbral_nulos", 0.6).definir("test_size", 0.2)

print("¿Son el mismo objeto?", a is b)
print("Semilla vista desde b:", b.semilla)
print("Parámetros desde b   :", b.parametros)

¿Son el mismo objeto? True
Semilla vista desde b: 42
Parámetros desde b   : {'umbral_nulos': 0.6, 'test_size': 0.2}


## Resumen: qué patrón usar y cuándo

| Patrón | El problema que resuelve | Dónde ya lo hicieron en la Fase 2 |
|---|---|---|
| **Strategy** | Varias formas de hacer lo mismo | Comparar imputaciones de `bmi` |
| **Factory** | Elegir qué construir según una condición | Decidir la transformación según el rol |
| **Observer** | Registrar sin que el ejecutor sepa quién anota | La bitácora de decisiones |
| **Singleton** | Una sola configuración compartida | La semilla y las rutas del proyecto |

**No usen los cuatro.** Usen el que su proyecto necesita y expliquen por qué. Un patrón
aplicado donde no hace falta agrega complejidad sin beneficio, y eso también se evalúa.

---
# 12. Adaptar el cuaderno a su propio conjunto

Hasta acá los pasos se escribieron a mano con los nombres de columna de stroke. Esta
sección construye el pipeline **desde la celda de configuración**, de modo que al
cambiar de proyecto no haya que tocar nada más que esa celda.

Es Factory aplicado al proyecto completo: la configuración deja de ser documentación y
pasa a gobernar el código.

In [ ]:
def construir_pipeline(escalar=True):
    """Arma el pipeline a partir de las columnas declaradas en la configuración.

    Orden de los pasos, y la razón de cada uno:
      1. eliminar el identificador, porque identifica y no describe
      2. imputar las continuas, antes de escalarlas
      3. codificar las nominales
      4. escalar las continuas, al final para que la imputación no altere la escala
    """
    pasos = []

    if COLUMNA_ID:
        pasos.append(EliminadorColumnas(COLUMNA_ID))

    for columna in COLUMNAS_CONTINUAS:
        pasos.append(ImputadorMediana(columna))

    for columna in COLUMNAS_NOMINALES:
        pasos.append(CodificadorNominal(columna))

    if escalar:
        for columna in COLUMNAS_CONTINUAS:
            pasos.append(EscaladorEstandar(columna))

    return Pipeline(pasos)


pipeline_config = construir_pipeline()
print(f"Pipeline construido con {len(pipeline_config)} pasos desde la configuración:\n")
print(pipeline_config.resumen().to_string(index=False))

Pipeline construido con 12 pasos desde la configuración:

 orden                                 paso              clase
     1               EliminadorColumnas(id) EliminadorColumnas
     2                ImputadorMediana(age)   ImputadorMediana
     3  ImputadorMediana(avg_glucose_level)   ImputadorMediana
     4                ImputadorMediana(bmi)   ImputadorMediana
     5           CodificadorNominal(gender) CodificadorNominal
     6        CodificadorNominal(work_type) CodificadorNominal
     7   CodificadorNominal(smoking_status) CodificadorNominal
     8   CodificadorNominal(Residence_type) CodificadorNominal
     9     CodificadorNominal(ever_married) CodificadorNominal
    10               EscaladorEstandar(age)  EscaladorEstandar
    11 EscaladorEstandar(avg_glucose_level)  EscaladorEstandar
    12               EscaladorEstandar(bmi)  EscaladorEstandar


In [ ]:
# Se ajusta con entrenamiento y se aplica a prueba, sin fuga de datos
pipeline_config.ajustar(entrenamiento)
entrenamiento_listo = pipeline_config.transformar(entrenamiento)
prueba_lista = pipeline_config.transformar(prueba)

print("Entrenamiento:", entrenamiento.shape, "->", entrenamiento_listo.shape)
print("Prueba       :", prueba.shape, "->", prueba_lista.shape)

# Las dos salidas deben tener exactamente las mismas columnas y en el mismo orden
assert list(entrenamiento_listo.columns) == list(prueba_lista.columns), \
    "Entrenamiento y prueba quedaron con columnas distintas"
print("\nAmbos conjuntos tienen las mismas columnas, en el mismo orden.")

Entrenamiento: (4088, 12) -> (4088, 22)
Prueba       : (1022, 12) -> (1022, 22)

Ambos conjuntos tienen las mismas columnas, en el mismo orden.


**Esa última comprobación es más importante de lo que parece.** Si el codificador
aprendiera el vocabulario de cada conjunto por separado, entrenamiento y prueba
quedarían con columnas distintas y cualquier modelo posterior fallaría. Como el
vocabulario se aprende una sola vez en el ajuste, eso no puede pasar.

## Guardar el resultado y su configuración

El conjunto procesado no sirve de mucho sin el registro de cómo se produjo. Guardar
ambos juntos es lo que hace el trabajo trazable.

In [ ]:
def guardar_resultado(df, ruta_salida, pipeline, carpeta="data/processed"):
    """Escribe el conjunto procesado y un registro de los parámetros aprendidos."""
    os.makedirs(carpeta, exist_ok=True)
    destino = os.path.join(carpeta, ruta_salida)
    df.to_csv(destino, index=False)

    registro = pd.DataFrame([
        {"orden": i, "paso": p.nombre, "parametros": str(p.parametros)}
        for i, p in enumerate(pipeline.pasos_ejecutados(), start=1)
    ])
    registro.to_csv(destino.replace(".csv", "_parametros.csv"), index=False)

    print(f"Conjunto guardado   : {destino} ({df.shape[0]} filas x {df.shape[1]} col)")
    print(f"Parámetros guardados: {destino.replace('.csv', '_parametros.csv')}")
    return registro


registro = guardar_resultado(entrenamiento_listo, "entrenamiento_procesado.csv",
                             pipeline_config)
registro.head()

Conjunto guardado   : data/processed/entrenamiento_procesado.csv (4088 filas x 22 col)
Parámetros guardados: data/processed/entrenamiento_procesado_parametros.csv


,orden,paso,parametros
0,1,EliminadorColumnas(id),{'existe': True}
1,2,ImputadorMediana(age),"{'mediana': 41.6, 'nulos_en_ajuste': 0}"
2,3,ImputadorMediana(avg_glucose_level),"{'mediana': 166.08499999999998, 'nulos_en_ajus..."
3,4,ImputadorMediana(bmi),"{'mediana': 28.7, 'nulos_en_ajuste': 163}"
4,5,CodificadorNominal(gender),"{'categorias': ['Female', 'Male', 'Other']}"


**Su turno.** Cambien la celda de configuración por las columnas de su propio conjunto
y vuelvan a ejecutar el cuaderno completo. Si todo está bien declarado, debería correr
sin modificar ninguna otra celda. Si falla, el mensaje de error les dice exactamente
qué columna no coincide.

---
# 13. El resultado: cómo queda el código y cómo quedan los datos

Esta sección cierra el recorrido. Muestra dos cosas: **cómo se ve el proyecto después de
aplicar POO** y **qué conjunto queda listo para la Fase 4**.

## 13.1 El código, antes y después

**Antes (Fase 2).** El pipeline era una secuencia de celdas. Funcionaba, pero el orden
estaba implícito, nada se podía reutilizar y la mediana se calculaba sobre el conjunto
completo:

```python
df = pd.read_csv("datos.csv")
df = df.drop(columns=["id"])
df["bmi"] = df["bmi"].fillna(df["bmi"].median())
df = pd.get_dummies(df, columns=["work_type"])
media = df["age"].mean(); desv = df["age"].std()
df["age"] = (df["age"] - media) / desv
```

**Después (Fase 3).** El mismo trabajo, en cuatro líneas que se leen como una
declaración de intenciones:

```python
pipeline = construir_pipeline()
pipeline.ajustar(entrenamiento)
entrenamiento_listo = pipeline.transformar(entrenamiento)
prueba_lista = pipeline.transformar(prueba)
```

Lo que cambió no es el resultado: es que ahora cada paso es una pieza con nombre, que se
puede probar, reemplazar y reutilizar sin tocar las demás.

## 13.2 La arquitectura, generada desde el propio código

En vez de escribir a mano la tabla de arquitectura para el informe, se puede producir
leyendo las clases que existen. Así no queda desactualizada.

In [ ]:
def documentar_arquitectura():
    """Genera la tabla de arquitectura a partir de las clases definidas.

    __subclasses__() devuelve las clases que heredan de Transformador. Es la
    misma información que Python usa para resolver la herencia, así que la
    tabla refleja el código real y no lo que creemos que hay.
    """
    filas = [{
        "componente": "Transformador",
        "rol": "clase base",
        "responsabilidad": "Define el contrato y controla el estado de cada paso",
        "archivo sugerido": "src/transformadores.py",
    }]

    for clase in Transformador.__subclasses__():
        resumen = (clase.__doc__ or "sin documentar").strip().split("\n")[0]
        filas.append({
            "componente": clase.__name__,
            "rol": "clase hija",
            "responsabilidad": resumen,
            "archivo sugerido": "src/transformadores.py",
        })

    filas += [
        {"componente": "Pipeline", "rol": "orquestador",
         "responsabilidad": "Encadena los pasos y controla el orden de ajuste",
         "archivo sugerido": "src/pipeline.py"},
        {"componente": "PipelineObservable", "rol": "orquestador",
         "responsabilidad": "Pipeline que además notifica cada paso a sus observadores",
         "archivo sugerido": "src/pipeline.py"},
        {"componente": "Bitacora", "rol": "observador",
         "responsabilidad": "Registra lo ocurrido en cada paso de la ejecución",
         "archivo sugerido": "src/observadores.py"},
        {"componente": "construir_pipeline", "rol": "fábrica",
         "responsabilidad": "Arma el pipeline desde la celda de configuración",
         "archivo sugerido": "src/fabrica.py"},
        {"componente": "medir", "rol": "utilidad",
         "responsabilidad": "Mide tiempo y memoria de cualquier función",
         "archivo sugerido": "src/medicion.py"},
    ]
    return pd.DataFrame(filas)


arquitectura = documentar_arquitectura()
print(arquitectura.to_string(index=False))

        componente         rol                                                   responsabilidad       archivo sugerido
     Transformador  clase base              Define el contrato y controla el estado de cada paso src/transformadores.py
  ImputadorMediana  clase hija                       Rellena los nulos con la mediana aprendida. src/transformadores.py
CodificadorNominal  clase hija              Convierte una columna de categorías en columnas 0/1. src/transformadores.py
 EscaladorEstandar  clase hija                         Centra en cero y escala a desviación uno. src/transformadores.py
EliminadorColumnas  clase hija Quita columnas que no aportan al análisis, como el identificador. src/transformadores.py
 ImputadorFlexible  clase hija    No sabe imputar: sabe cuándo. El cómo lo aporta la estrategia. src/transformadores.py
          Pipeline orquestador                  Encadena los pasos y controla el orden de ajuste        src/pipeline.py
PipelineObservable orquestador         P

**Esa tabla es el apartado de documentación de arquitectura del informe.** Cópienla,
agreguen dos o tres párrafos explicando **por qué** esa división, y el criterio queda
cubierto.

La estructura de archivos que sugiere la última columna es la que corresponde llevar al
repositorio:

```
F3/
├── notebooks/
│   └── f3_s02_guia_poo.ipynb      este cuaderno, ejecutado
├── src/
│   ├── transformadores.py          la clase base y sus hijas
│   ├── pipeline.py                 Pipeline y PipelineObservable
│   ├── observadores.py             Bitacora y ReporteConsola
│   ├── fabrica.py                  construir_pipeline()
│   └── medicion.py                 medir() y comparaciones
└── data/processed/                 lo que se entrega a la Fase 4
```

## 13.3 El conjunto procesado: qué queda listo

Acá se ejecuta el pipeline completo y se revisa el resultado con el detalle que la
Fase 4 va a necesitar.

In [ ]:
# Pipeline definitivo, construido desde la configuración
pipeline_final = construir_pipeline()
pipeline_final.ajustar(entrenamiento)

entrenamiento_final = pipeline_final.transformar(entrenamiento)
prueba_final = pipeline_final.transformar(prueba)

print("CONJUNTO PROCESADO")
print("=" * 58)
print(f"Entrenamiento : {entrenamiento_final.shape[0]:>6} filas x {entrenamiento_final.shape[1]:>3} columnas")
print(f"Prueba        : {prueba_final.shape[0]:>6} filas x {prueba_final.shape[1]:>3} columnas")
print(f"Partida desde : {datos.shape[0]:>6} filas x {datos.shape[1]:>3} columnas")
print(f"\nColumnas ganadas por la codificación: "
      f"{entrenamiento_final.shape[1] - datos.shape[1] + 1}")
entrenamiento_final.head(3)

CONJUNTO PROCESADO
Entrenamiento :   4088 filas x  22 columnas
Prueba        :   1022 filas x  22 columnas
Partida desde :   5110 filas x  12 columnas

Columnas ganadas por la codificación: 11


,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke,gender_Female,gender_Male,gender_Other,work_type_Govt_job,...,work_type_Self_employed,work_type_children,smoking_status_Unknown,smoking_status_formerly_smoked,smoking_status_never_smoked,smoking_status_smokes,Residence_type_Rural,Residence_type_Urban,ever_married_No,ever_married_Yes
4688,0.410470,0,0,1.315090,0.671446,0,1,0,0,0,...,0,0,0,0,1,0,1,0,0,1
4478,-1.211171,0,0,-1.643162,1.059149,0,0,1,0,1,...,0,0,0,1,0,0,1,0,0,1
3849,-0.817404,0,0,1.385363,1.511469,0,0,1,0,0,...,0,1,0,0,0,1,1,0,0,1


In [ ]:
# Esquema final: qué tipo tiene cada columna y de dónde salió
def describir_esquema(df, originales):
    """Clasifica cada columna del resultado según su origen."""
    filas = []
    for columna in df.columns:
        if columna in originales:
            origen = "original"
        elif "_" in columna and columna.split("_")[0] in originales:
            origen = f"derivada de {columna.split('_')[0]}"
        else:
            origen = "derivada"
        filas.append({
            "columna": columna,
            "tipo": str(df[columna].dtype),
            "origen": origen,
            "nulos": int(df[columna].isna().sum()),
        })
    return pd.DataFrame(filas)


esquema = describir_esquema(entrenamiento_final, set(datos.columns))
print(f"Total de columnas: {len(esquema)}")
print(f"Originales conservadas: {(esquema['origen'] == 'original').sum()}")
print(f"Derivadas de la codificación: {(esquema['origen'] != 'original').sum()}")
print(f"Columnas con nulos: {(esquema['nulos'] > 0).sum()}")
print()
esquema.head(12)

Total de columnas: 22
Originales conservadas: 6
Derivadas de la codificación: 16
Columnas con nulos: 0



,columna,tipo,origen,nulos
0,age,float64,original,0
1,hypertension,int64,original,0
2,heart_disease,int64,original,0
3,avg_glucose_level,float64,original,0
4,bmi,float64,original,0
5,stroke,int64,original,0
6,gender_Female,int64,derivada de gender,0
7,gender_Male,int64,derivada de gender,0
8,gender_Other,int64,derivada de gender,0
9,work_type_Govt_job,int64,derivada,0


## 13.4 Verificación de entrega

Antes de pasar a la Fase 4 conviene comprobar cinco condiciones. Si alguna falla, el
problema es de esta fase y corregirlo acá cuesta mucho menos que descubrirlo después.

In [ ]:
def verificar_entrega(entrenamiento_final, prueba_final, datos_originales, objetivo):
    """Cinco comprobaciones antes de entregar el conjunto a la Fase 4."""
    controles = []

    # 1. No se perdieron ni se duplicaron filas
    esperadas = len(datos_originales)
    obtenidas = len(entrenamiento_final) + len(prueba_final)
    controles.append(("Se conservan todas las filas",
                      obtenidas == esperadas, f"{obtenidas} de {esperadas}"))

    # 2. Entrenamiento y prueba tienen el mismo esquema
    mismo_esquema = list(entrenamiento_final.columns) == list(prueba_final.columns)
    controles.append(("Mismo esquema en ambos conjuntos",
                      mismo_esquema, f"{entrenamiento_final.shape[1]} columnas"))

    # 3. No quedan nulos en las columnas continuas tratadas
    nulos = int(entrenamiento_final[COLUMNAS_CONTINUAS].isna().sum().sum())
    controles.append(("Sin nulos en las continuas tratadas",
                      nulos == 0, f"{nulos} nulos"))

    # 4. La variable objetivo sobrevivió intacta
    objetivo_ok = (objetivo in entrenamiento_final.columns
                   and entrenamiento_final[objetivo].isna().sum() == 0)
    controles.append(("Variable objetivo presente y completa",
                      objetivo_ok, objetivo))

    # 5. El identificador fue eliminado
    sin_id = COLUMNA_ID not in entrenamiento_final.columns
    controles.append(("Identificador eliminado", sin_id, COLUMNA_ID))

    tabla = pd.DataFrame(
        [{"control": c[0], "estado": "OK" if c[1] else "FALLA", "detalle": c[2]}
         for c in controles]
    )
    aprobado = all(c[1] for c in controles)
    return tabla, aprobado


controles, aprobado = verificar_entrega(entrenamiento_final, prueba_final,
                                        datos, COLUMNA_OBJETIVO)
print(controles.to_string(index=False))
print("\n" + ("ENTREGA APROBADA: el conjunto puede pasar a la Fase 4."
               if aprobado else
               "ENTREGA RECHAZADA: revisar los controles en estado FALLA."))

                              control estado      detalle
         Se conservan todas las filas     OK 5110 de 5110
     Mismo esquema en ambos conjuntos     OK  22 columnas
  Sin nulos en las continuas tratadas     OK      0 nulos
Variable objetivo presente y completa     OK       stroke
              Identificador eliminado     OK           id

ENTREGA APROBADA: el conjunto puede pasar a la Fase 4.


## 13.5 Lo que se entrega a la Fase 4

La Fase 4 no debe volver a limpiar ni transformar. Recibe cuatro artefactos y trabaja
sobre ellos.

In [ ]:
def entregar_a_fase4(entrenamiento_final, prueba_final, pipeline, esquema,
                     carpeta="data/processed"):
    """Escribe los cuatro artefactos que recibe la Fase 4."""
    os.makedirs(carpeta, exist_ok=True)
    entregables = {}

    # 1. Los conjuntos listos para analizar
    ruta_ent = os.path.join(carpeta, "entrenamiento_f3.csv")
    ruta_pru = os.path.join(carpeta, "prueba_f3.csv")
    entrenamiento_final.to_csv(ruta_ent, index=False)
    prueba_final.to_csv(ruta_pru, index=False)
    entregables["entrenamiento"] = ruta_ent
    entregables["prueba"] = ruta_pru

    # 2. El diccionario del conjunto resultante
    ruta_dic = os.path.join(carpeta, "diccionario_f3.csv")
    esquema.to_csv(ruta_dic, index=False)
    entregables["diccionario"] = ruta_dic

    # 3. Los parámetros aprendidos: es lo que permite reproducir el resultado
    ruta_par = os.path.join(carpeta, "parametros_f3.csv")
    pd.DataFrame([
        {"orden": i, "paso": p.nombre, "parametros": str(p.parametros)}
        for i, p in enumerate(pipeline.pasos_ejecutados(), start=1)
    ]).to_csv(ruta_par, index=False)
    entregables["parametros"] = ruta_par

    return entregables


entregables = entregar_a_fase4(entrenamiento_final, prueba_final,
                               pipeline_final, esquema)

print("ARTEFACTOS ENTREGADOS A LA FASE 4")
print("=" * 58)
for nombre, ruta in entregables.items():
    tamano = os.path.getsize(ruta) / 1024
    print(f"  {nombre:<16} {ruta:<40} {tamano:>7.1f} KB")

ARTEFACTOS ENTREGADOS A LA FASE 4
  entrenamiento    data/processed/entrenamiento_f3.csv        386.5 KB
  prueba           data/processed/prueba_f3.csv                97.0 KB
  diccionario      data/processed/diccionario_f3.csv            0.8 KB
  parametros       data/processed/parametros_f3.csv             1.0 KB


### El contrato entre fases

| Artefacto | Qué contiene | Para qué lo usa la Fase 4 |
|---|---|---|
| `entrenamiento_f3.csv` | El conjunto procesado de entrenamiento | Ajustar y explorar |
| `prueba_f3.csv` | El conjunto procesado de prueba | Evaluar, con el mismo esquema |
| `diccionario_f3.csv` | Columna, tipo, origen y nulos | Saber qué significa cada variable |
| `parametros_f3.csv` | Lo que aprendió cada paso | Reproducir el resultado o auditarlo |

**Lo que la Fase 4 NO debe hacer:** volver a imputar, volver a codificar ni volver a
escalar. Si lo hiciera sobre el conjunto ya transformado, escalaría dos veces y los
resultados dejarían de tener sentido.

**Lo que sí debe hacer:** leer estos archivos, explorar, visualizar y comunicar. Y si
necesita una transformación nueva, agregarla como un paso más del pipeline de la Fase 3,
no como código suelto en su propio cuaderno. Esa es la ventaja de haber modularizado.

In [ ]:
# Comprobación final: lo que se guardó se puede volver a leer sin sorpresas
releido = pd.read_csv(entregables["entrenamiento"])

print("Verificación de ida y vuelta")
print(f"  Antes de guardar : {entrenamiento_final.shape}")
print(f"  Después de leer  : {releido.shape}")
print(f"  Mismas columnas  : {list(releido.columns) == list(entrenamiento_final.columns)}")
print(f"  Sin nulos nuevos : {int(releido.isna().sum().sum()) == int(entrenamiento_final.isna().sum().sum())}")
print("\nEl conjunto está listo para la Fase 4.")

Verificación de ida y vuelta
  Antes de guardar : (4088, 22)
  Después de leer  : (4088, 22)
  Mismas columnas  : True
  Sin nulos nuevos : True

El conjunto está listo para la Fase 4.


---
# 14. Su turno: ejercicios para el proyecto

Estos cinco ejercicios producen directamente la evidencia que pide la rúbrica.
Resuélvanlos con **su propio conjunto de datos**, no con el de stroke.

## Ejercicio 1 · Encapsular un paso de su pipeline
**Alimenta:** criterio de programación orientada a objetos (6 pts).

In [1]:
class MiTransformador(Transformador):
    """TODO: describir qué hace, sobre qué columna y por qué."""

    def aprender(self, df):
        # TODO: calcular lo que hay que aprender del conjunto de entrenamiento
        # Devuelve un diccionario con los parámetros
        return {}

    def aplicar(self, df):
        # TODO: aplicar la transformación usando self._parametros
        return df


# TODO: probarla con su conjunto
# mi_paso = MiTransformador("su_columna")
# resultado = mi_paso.ajustar_transformar(su_df)
# print(mi_paso.nombre, mi_paso.parametros)

NameError: name 'Transformador' is not defined

## Ejercicio 2 · Armar su pipeline completo
**Alimenta:** diseño estructurado (6 pts) y preprocesamiento (6 pts).

In [ ]:
# TODO: reemplazar por los pasos de SU pipeline de la Fase 2
mi_pipeline = Pipeline([
    # EliminadorColumnas("su_identificador"),
    # ImputadorFlexible("su_columna_con_nulos"),
    # CodificadorNominal("su_categorica"),
    # EscaladorEstandar("su_continua"),
])

# TODO: ajustar con entrenamiento y transformar prueba
# entrenamiento = su_df.sample(frac=0.8, random_state=SEMILLA)
# prueba = su_df.drop(index=entrenamiento.index)
# mi_pipeline.ajustar(entrenamiento)
# resultado = mi_pipeline.transformar(prueba)

## Ejercicio 3 · Comparar dos implementaciones
**Alimenta:** eficiencia y optimización (6 pts).

In [ ]:
def implementacion_a(df):
    # TODO: una forma, por ejemplo con bucle sobre filas
    pass


def implementacion_b(df):
    # TODO: otra forma, por ejemplo vectorizada
    pass


# TODO: comprobar primero que dan el mismo resultado
# assert implementacion_a(su_df) == implementacion_b(su_df)

# TODO: medir con timeit y con medir() para la memoria
# TODO: repetir sobre varios tamaños para ver cómo crece la diferencia
# TODO: interpretar el resultado en una celda de texto

## Ejercicio 4 · Validar en tres escenarios
**Alimenta:** validación técnica (6 pts).

In [ ]:
# TODO: CASO NORMAL
# assert len(resultado) == len(su_df), "El pipeline perdió o duplicó filas"

# TODO: CASO LÍMITE
# Ideas: columna sin nulos, una sola categoría, varianza cero,
#        categoría que aparece solo en prueba

# TODO: EXCEPCIÓN
# try:
#     MiTransformador("columna_inexistente").ajustar(su_df)
# except KeyError as error:
#     print("KeyError capturado:", error)

## Ejercicio 5 · Elegir un patrón y verificar que nada se rompió
**Alimenta:** documentación de arquitectura (6 pts) y validación técnica (6 pts).

In [ ]:
# TODO 1: elegir UN patrón de la tabla resumen e implementarlo.
#         Escribir tres frases: qué problema resolvía, cómo lo resuelve
#         el patrón, y qué habría pasado sin él.

# TODO 2: comparar el conjunto procesado de la Fase 2 contra el que
#         produce su pipeline de clases
# anterior = pd.read_csv("../F2/data/processed/su_archivo.csv")
# nuevo = mi_pipeline.ajustar(su_df).transformar(su_df)
# pd.testing.assert_frame_equal(anterior, nuevo)
# print("La reorganización no alteró el resultado")

---
# Cierre

## Lo que se llevan

| Concepto | Dónde se ve en este cuaderno |
|---|---|
| Encapsulamiento | `_parametros`, `_ajustado`, propiedades de solo lectura |
| Herencia | `ImputadorMediana`, `CodificadorNominal` y `EscaladorEstandar` heredan de `Transformador` |
| Polimorfismo | El `Pipeline` recorre los pasos sin preguntar su tipo |
| Recursividad | `aplanar()` y el contraste entre Fibonacci ingenua y memoizada |
| Eficiencia | `timeit`, `perf_counter`, `tracemalloc` y la tabla de crecimiento |
| Patrones | Strategy, Factory, Observer y Singleton |

## Antes de la próxima sesión

1. Conviertan **un** paso de su pipeline en una clase que herede de `Transformador`.
2. Midan **una** operación de dos formas distintas, con la comprobación de equivalencia.
3. Decidan si su proyecto justifica recursividad, y anoten por qué.
4. Ejecuten `assert_frame_equal` contra su conjunto de la Fase 2.

Con esos cuatro pasos tienen el esqueleto de la entrega.

## Bibliografía (APA 7)

Gamma, E., Helm, R., Johnson, R., & Vlissides, J. (1994). *Design patterns: Elements of
reusable object-oriented software*. Addison-Wesley.

McKinney, W. (2022). *Python for data analysis* (3.ª ed.). O'Reilly Media.

Python Software Foundation. (s. f.). *Classes*. https://docs.python.org/3/tutorial/classes.html

Python Software Foundation. (s. f.). *timeit — Measure execution time of small code
snippets*. https://docs.python.org/3/library/timeit.html

The pandas development team. (s. f.). *pandas documentation*. https://pandas.pydata.org/docs/

---

*Avance Fase 3 · Semana 2 · MCDI500 · Magíster en Ciencia de Datos e Inteligencia
Artificial · Universidad Andrés Bello*